# 🔬 Unified Narrative Intelligence Pipeline
### Run once → CAO persisted → Tier 1 / 2 / 3 regenerate instantly

```
┌─────────────────────────────────────────────────────────────────────┐
│  PHASE A — PROCESS ONCE                                             │
│  PDF → Chunks → Embed → Topics → Clusters → Annotate → NLI          │
│                          ↓                                          │
│              canonical_analysis_object.parquet  (CAO)               │
│              cao_meta.json  (fingerprint + patterns)                │
│                                                                     │
│  PHASE B — REPORT ANYTIME (no reprocessing)                         │
│  CAO → Tier 1  filtered high-signal patterns     (t1_report.docx)   │
│      → Tier 2  expanded context + targets        (t2_report.docx)   │
│      → Tier 3  full CAO + NLI + network          (t3_report.docx)   │
└─────────────────────────────────────────────────────────────────────┘
```

**Required packages:**
```
pip install pdfplumber spacy sentence-transformers bertopic umap-learn hdbscan \
            networkx python-docx openpyxl pandas pyarrow matplotlib transformers \
            nltk scikit-learn
python -m spacy download en_core_web_sm
```
### Includes Word document generation at end of run
### Updated Word Chunking self aware testing in Phase A 4. Ingest & Chunk

---
## 0 · Imports & Logging

In [ ]:
import os, re, json, logging, warnings
from datetime import datetime
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
import nltk
import spacy
import pdfplumber
from docx import Document
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from transformers import pipeline as hf_pipeline

warnings.filterwarnings("ignore")
for pkg in ["punkt", "punkt_tab", "stopwords"]:
    nltk.download(pkg, quiet=True)

nlp = spacy.load("en_core_web_sm")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)
log.info("✅ Imports complete")

---
## 1 · Configuration
Set your PDF path and tuning parameters here. Everything else is automatic.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ❶  SET YOUR PDF PATH
# ──────────────────────────────────────────────────────────────────────────────
PDF_PATH   =                                 # ← change this
BASE_OUTPUT_DIR = "outputs"                  # root output folder — never overwritten

# ──────────────────────────────────────────────────────────────────────────────
# ❶b RUN IDENTIFIER  — fills OUTPUT_DIR automatically, never overwrites
# ──────────────────────────────────────────────────────────────────────────────
CLIENT_NAME =                               # ← e.g. "Test1", "AcmeCorp", "ClientA"

_doc_name   = Path(PDF_PATH).stem           # filename without extension
_timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
_parts      = [p for p in [CLIENT_NAME, _doc_name, _timestamp] if p]
OUTPUT_DIR  = os.path.join(BASE_OUTPUT_DIR, "_".join(_parts))

# ──────────────────────────────────────────────────────────────────────────────
# ❶c PAGE RANGE  (set both to None to process the entire PDF)
# ──────────────────────────────────────────────────────────────────────────────
PAGE_START = 1    # first page to include (1-indexed), e.g. 45
PAGE_END   = 21    # last page to include  (1-indexed, inclusive), e.g. 112

# ──────────────────────────────────────────────────────────────────────────────
# ❷  PIPELINE TUNING
# ──────────────────────────────────────────────────────────────────────────────
CHUNK_SIZE      = 400
N_CLUSTERS      = 10   # desired number of clusters; auto-clamped later if doc is short
NR_TOPICS       = "auto"
NLI_TOP_N       = 50
NLI_THRESHOLD   = 0.02
# ──────────────────────────────────────────────────────────────────────────────
# ❸  TIER FILTERS  (change at any time — no reprocessing needed)
# ──────────────────────────────────────────────────────────────────────────────
TIER1_MIN_LEX_SUM  = 0.10
TIER1_MIN_NLI      = 0.50
TIER2_MIN_LEX_SUM  = 0.05
TIER2_TOP_TARGETS  = 30
# Tier 3 = full CAO — no filter
# ──────────────────────────────────────────────────────────────────────────────
# ❹  CAO FILE PATHS  (auto-named; override if needed)
# ──────────────────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
CAO_PARQUET = os.path.join(OUTPUT_DIR, "canonical_analysis_object.parquet")
CAO_META    = os.path.join(OUTPUT_DIR, "cao_meta.json")
log.info("Run ID:  %s", os.path.basename(OUTPUT_DIR))
log.info("Output:  %s", OUTPUT_DIR)
log.info("Page range: %s", f"{PAGE_START}–{PAGE_END}" if (PAGE_START or PAGE_END) else "full document")

---
## 2 · Lexicon, NLI Hypotheses & Constants
**6 rhetorical dimensions** — sourced from `Text_Analysis_Research_2_4`.

In [ ]:
LEXICON: dict[str, set[str]] = {
    "moral": {
        "character","conscience","conviction","dignity","duty","ethic",
        "excellence","faith","honor","honour","integrity","moral","principle",
        "responsibility","righteousness","virtue","worth","worthiness",
        "child","children","community","family","father","household","marriage",
        "mother","parent","parenthood","biblical","blessing","church","creator",
        "divine","doctrine","god","gospel","grace","holy","prayer","providence",
        "religion","religious","sacred","sanctity","spiritual","american",
        "ancestry","citizen","civilization","civilisation","constitutional",
        "culture","founding","freedom","heritage","history","identity","legacy",
        "liberty","nation","national","patriot","patriotism","republic",
        "sovereign","sovereignty","tradition","traditional","value","decay",
        "decency","depravity","degenerate","immoral","indecent","pornography",
        "promiscuity","restore","revival","wholesome","justice","fair","right",
        "wrong","ethical","honest","good","evil","compassion","Compassionate",
        "care","kind","selfish","greedy","corrupt","innocent","guilty","sacred",
    },
    "threat": {
        "attack","catastrophe","chaos","collapse","crisis","danger",
        "destabilize","destabilise","devastate","disaster","emergency",
        "existential","exploit","hazard","jeopardize","jeopardise","peril",
        "risk","threat","vulnerability","abuse","corruption","corrupt",
        "illegitimate","infiltrate","manipulation","misconduct","subvert",
        "subversion","undermine","weaponize","weaponise","adversary",
        "authoritarian","communist","enemy","extremist","ideologue",
        "indoctrinate","marxist","radical","socialist","woke","bloated",
        "broken","bureaucratic","dysfunction","dysfunctional","entrenched",
        "failure","incompetent","inefficient","neglect","obstruction",
        "overreach","unaccountable","wasteful","erode","erosion","decline",
        "degradation","deterioration","disintegration","risk","destroy","damage",
        "harm","fear","warning","vulnerable","weak","hostile","invade","loss",
        "break","fail","prevent","block","stop","hazard",
    },
    "power": {
        "accountability","administer","administration","appoint","authority",
        "centralize","centralise","command","commissioner","consolidate",
        "control","coordinate","delegate","department","designate","direct",
        "director","dismiss","enforce","executive","fire","govern","governance",
        "head","hierarchy","implement","jurisdiction","manage","mandate",
        "oversight","power","presidential","principal","remove","reorganize",
        "reorganise","require","restructure","secretariat","secretary",
        "subordinate","supervise","terminate","ban","compliance","compel",
        "constrain","constraint","enforcement","forbid","impose","prohibit",
        "prohibition","regulate","regulation","restriction","rule","sanction",
        "allocate","appropriate","budget","defund","funding","grant",
        "privatize","privatise","resource","transfer","dominate","force","strong",
        "lead","influence","decide","determine","execute","rule","sovereign",
        "overcome","triumph","crush","defeat","superior","might",
    },
    "urgency": {
        "asap","demand","expedite","forthwith","immediate","immediately",
        "imminent","imperative","instantaneous","now","presently","prompt",
        "promptly","quick","rapid","rapidly","soon","swift","swiftly","urgent",
        "urgently","cannot","compel","compulsory","critical","crucial",
        "essential","indispensable","mandatory","must","necessary","necessity",
        "need","nonnegotiable","obligate","obligation","require","requirement",
        "shall","vital","before","deadline","delay","expire","first","initial",
        "moment","opportunity","priority","transition","window","catastrophic",
        "decisive","historic","landmark","once","unprecedented","hurry","after",
        "finally","emergency","pressing","time","sudden","accelerate","decelerate",
    },
    "us_vs_them": {
        "ally","american","coalition","colleague","compatriot","conservative",
        "constitutional","fellow","friend","loyal","movement","our","partner",
        "patriot","people","supporter","team","together","unified","unity",
        "us","we","adversary","alien","anarchist","antiamerican","bureaucrat",
        "deep","elite","enemy","establishment","extremist","faction","foreign",
        "globalist","hostile","ideologue","infiltrator","liberal","leftist",
        "marxist","opponent","opposition","outsider","radical","regime",
        "socialist","subversive","swamp","them","they","unelected","woke",
        "battle","combat","confrontation","contest","defeat","defend","fight",
        "resist","resistance","struggle","war","warfare",
    },
    "legitimacy": {
        "amendment","article","charter","clause","codify","congress",
        "constitution","constitutional","court","enact","enumerate","founding",
        "frame","framers","judicial","jurisdiction","law","lawful","legal",
        "legislation","legitimate","precedent","provision","ratify","statute",
        "statutory","accountability","ballot","bipartisan","check","citizen",
        "civic","civil","consent","democratic","election","elect","mandate",
        "public","referendum","represent","representation","transparent",
        "transparency","vote","voter","agency","bureau","commission",
        "committee","department","expert","federal","institutional","office",
        "official","policy","process","professional","protocol","standard",
        "illegitimate","unconstitutional","unlawful","unelected",
        "unaccountable","extralegal",
    },
}
LEXICON_FROZEN: dict[str, frozenset[str]] = {
    dim: frozenset(terms) for dim, terms in LEXICON.items()
}
DIMENSIONS: list[str] = list(LEXICON.keys())

NLI_HYPOTHESES: dict[str, str] = {
    "moral":      "This text appeals to moral values, religious beliefs, or traditional cultural identity.",
    "threat":     "This text portrays a situation, group, or institution as dangerous, corrupt, or threatening.",
    "power":      "This text is concerned with authority, control, enforcement, or the exercise of governmental power.",
    "urgency":    "This text conveys a sense of urgency, immediate necessity, or time-critical action.",
    "us_vs_them": "This text constructs an in-group versus out-group distinction or frames politics as a conflict between opposing sides.",
    "legitimacy": "This text invokes legal authority, constitutional principles, democratic mandate, or institutional legitimacy.",
}

TARGET_VERBS = frozenset([
    "abolish","control","create","cut","defund","eliminate","empower",
    "enforce","expand","mandate","privatise","privatize","protect",
    "reduce","reform","regulate","restructure","strengthen","transfer",
])

SPACY_BATCH      = 350
SENTIMENT_BATCH  = 128
log.info("Lexicon loaded: %d dimensions, %d total terms",
         len(DIMENSIONS), sum(len(v) for v in LEXICON.values()))

---
## 3 · Model Initialisation
Models are loaded once here and reused throughout.

In [ ]:
log.info("Loading DistilBERT sentiment model…")
_sentiment_pipeline = hf_pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    truncation=True,
    max_length=512,
)

_nli_pipeline = None  # lazy-loaded in Phase A step 8

def _get_nli_pipeline():
    global _nli_pipeline
    if _nli_pipeline is None:
        log.info("Loading NLI model (facebook/bart-large-mnli) — first call only…")
        _nli_pipeline = hf_pipeline(
            "zero-shot-classification",
            model="facebook/bart-large-mnli",
            device=-1,
            multi_label=True,
        )
        log.info("NLI model ready.")
    return _nli_pipeline

def _batch_sentiment(entity_texts: list[str]) -> list[dict]:
    if not entity_texts:
        return []
    return _sentiment_pipeline(entity_texts, batch_size=SENTIMENT_BATCH)

log.info("✅ Sentiment model loaded.")

---
# ━━━  PHASE A — PROCESS ONCE  ━━━
Run cells 4–10 exactly once. They produce the **Canonical Analysis Object** (CAO).
After that, jump directly to Phase B to regenerate any tier report instantly.

---
## 4 · Ingest & Chunk

In [ ]:
def load_pdf(path: str) -> list[str]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF not found: {path}")

    pages = []
    with pdfplumber.open(path) as pdf:
        # Resolve page range (convert 1-indexed config values to 0-indexed slice)
        start = (PAGE_START - 1) if PAGE_START is not None else 0
        end   = PAGE_END          if PAGE_END   is not None else len(pdf.pages)

        for page in pdf.pages[start:end]:
            text = page.extract_text()
            if text and text.strip():
                pages.append(text)

    log.info("Loaded %d pages from '%s' (pages %s–%s)",
             len(pages), path.name,
             PAGE_START or 1, PAGE_END or "end")
    return pages

def remove_headers_footers(pages: list[str], threshold: float = 0.30) -> list[str]:
    n = len(pages)

    # ── Scale-aware threshold ──────────────────────────────────────────────────
    # A flat 30% cutoff over-strips short documents (e.g. 3 pages → cutoff=0.9,
    # meaning ANY line that appears more than once is removed).
    # The table below shows the effective threshold by document length:
    #   ≤  5 pages  →  0.90  (must appear on 90% of pages to be stripped)
    #   ≤ 10 pages  →  0.75
    #   ≤ 20 pages  →  0.60
    #   ≤ 50 pages  →  0.45
    #    > 50 pages  →  use caller-supplied threshold (default 0.30)
    if n <= 5:
        effective_threshold = 0.90
    elif n <= 10:
        effective_threshold = 0.75
    elif n <= 20:
        effective_threshold = 0.60
    elif n <= 50:
        effective_threshold = 0.45
    else:
        effective_threshold = threshold  # original behaviour for long docs

    cutoff = n * effective_threshold

    log.info(
        "Header/footer threshold: %.2f (effective=%.2f, doc=%d pages)",
        threshold, effective_threshold, n,
    )

    # ── Count line frequency across all pages ─────────────────────────────────
    line_counts: Counter = Counter()
    for page in pages:
        for line in set(page.split("\n")):
            stripped = line.strip()
            if stripped:
                line_counts[stripped] += 1

    repeating = {line for line, count in line_counts.items() if count > cutoff}

    # ── Strip repeating lines and warn if too much content was removed ─────────
    cleaned = []
    total_lines_in  = 0
    total_lines_out = 0
    for page in pages:
        all_lines = page.split("\n")
        kept = [l for l in all_lines if l.strip() not in repeating]
        cleaned.append("\n".join(kept))
        total_lines_in  += len(all_lines)
        total_lines_out += len(kept)

    lines_removed = total_lines_in - total_lines_out
    pct_removed   = (lines_removed / total_lines_in * 100) if total_lines_in else 0.0

    log.info(
        "Removed %d repeating header/footer lines (%.1f%% of all lines).",
        len(repeating), pct_removed,
    )

    # Guard: if more than 60% of all lines were stripped, something is wrong —
    # fall back to returning pages unmodified so downstream steps aren't starved.
    if pct_removed > 60.0:
        log.warning(
            "Header/footer removal wiped %.1f%% of content (threshold may be too "
            "aggressive for a %d-page document). Returning original pages unmodified.",
            pct_removed, n,
        )
        return pages

    return cleaned

def chunk_text(pages: list[str], max_chars: int = 400) -> list[str]:
    chunks = []
    for page in pages:
        sentences = nltk.sent_tokenize(re.sub(r"\s+", " ", page).strip().lower())
        current = ""
        for sent in sentences:
            if len(current) + len(sent) < max_chars:
                current = (current + " " + sent).strip()
            else:
                if current:
                    chunks.append(current)
                current = sent
        if current:
            chunks.append(current)
    chunks = [c for c in chunks if len(c) > 20]
    log.info("Created %d text chunks.", len(chunks))
    return chunks

# ── RUN ──
pages  = load_pdf(PDF_PATH)
pages  = remove_headers_footers(pages)
chunks = chunk_text(pages, max_chars=CHUNK_SIZE)

---
## 5 · Embed + Optional Elbow Diagnostic

In [ ]:
def embed(chunks: list[str], model_name: str = "all-MiniLM-L6-v2") -> np.ndarray:
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=True, batch_size=64)
    log.info("Embeddings shape: %s", embeddings.shape)
    return embeddings

embeddings = embed(chunks)

In [ ]:
# ── OPTIONAL: Elbow diagnostic to choose N_CLUSTERS ──────────────────────────
# Run this cell BEFORE cell 6 if you want data-driven cluster selection.
# Set N_CLUSTERS = <elbow value> in cell 1 then re-run from cell 6 onwards.

_max_k = min(15, len(embeddings) - 1)   # never ask for more clusters than samples

if _max_k < 4:
    log.warning(
        "Only %d chunks available — too few for a meaningful elbow plot. "
        "Skipping diagnostic; N_CLUSTERS will be clamped in Cell 6.", len(embeddings)
    )
else:
    _k_range  = range(2, _max_k + 1, 2)
    _inertias = []
    for _k in _k_range:
        _km = KMeans(n_clusters=_k, random_state=42, n_init="auto")
        _km.fit(embeddings)
        _inertias.append(_km.inertia_)

    _ks   = list(_k_range)
    _d2   = [(_inertias[i]-_inertias[i+1]) - (_inertias[i+1]-_inertias[i+2])
              for i in range(len(_inertias)-2)]
    _elbow = _ks[int(np.argmax(_d2)) + 1]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(_ks, _inertias, marker="o", linewidth=2)
    ax.axvline(_elbow, color="green", linestyle="--", label=f"elbow k={_elbow}")
    ax.set(xlabel="k", ylabel="inertia", title="Elbow Diagnostic")
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f"Suggested N_CLUSTERS = {_elbow}")

---
## 6 · Topics & Clusters

In [ ]:
def run_topic_model(chunks, embeddings, nr_topics="auto"):
    # BERTopic needs enough docs to form meaningful topics.
    # With small corpora, "auto" reduction can collapse everything to 0 topics.
    # Force a safe upper bound: at most 1 topic per 3 chunks, minimum 2.
    _max_topics = max(2, len(chunks) // 3)
    safe_nr_topics = _max_topics if nr_topics == "auto" else min(nr_topics, _max_topics)

    if safe_nr_topics != nr_topics:
        log.warning(
            "NR_TOPICS changed from %r → %d (corpus has only %d chunks).",
            nr_topics, safe_nr_topics, len(chunks),
        )

    model = BERTopic(nr_topics=safe_nr_topics, verbose=False)
    topics, _ = model.fit_transform(chunks, embeddings)
    log.info("Discovered %d topics.", len(model.get_topic_info()))
    return model, topics


def cluster(embeddings, n_clusters=10, random_state=42):
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto")
    labels = km.fit_predict(embeddings)
    log.info("Cluster distribution: %s", dict(Counter(labels.tolist())))
    return labels


topic_model_obj, topics = run_topic_model(chunks, embeddings, NR_TOPICS)

# Clamp N_CLUSTERS so it never exceeds available samples
_n_clusters = min(N_CLUSTERS, len(embeddings) - 1)
if _n_clusters < N_CLUSTERS:
    log.warning(
        "N_CLUSTERS clamped from %d → %d (only %d chunks available).",
        N_CLUSTERS, _n_clusters, len(embeddings),
    )
cluster_labels = cluster(embeddings, _n_clusters)

---
## 7 · Annotate — 3-Pass Vectorised Build
Single spaCy pipe → batched DistilBERT → DataFrame assembly.

In [ ]:
def build_dataset(chunks, topics, clusters) -> pd.DataFrame:
    total = len(chunks)
    log.info("Pass 1/3 — spaCy pipe (%d chunks, batch=%d)…", total, SPACY_BATCH)

    all_entities   = [None] * total
    all_targets    = [None] * total
    all_lex        = [None] * total
    entity_ranges  = [None] * total
    flat_ent_texts = []

    for i, doc in enumerate(nlp.pipe(chunks, batch_size=SPACY_BATCH)):
        if i % 200 == 0:
            log.info("  spaCy: %d / %d", i, total)

        chunk_ents = [e.text for e in doc.ents
                      if e.label_ in {"ORG","PERSON","GPE","LAW","NORP"}]
        start = len(flat_ent_texts)
        flat_ent_texts.extend(chunk_ents)
        entity_ranges[i] = (start, start + len(chunk_ents))

        ent_dict = {k: [] for k in ("ORG","PERSON","GPE","LAW","NORP")}
        for e in doc.ents:
            if e.label_ in ent_dict:
                ent_dict[e.label_].append(e.text)
        all_entities[i] = {k: sorted(set(v)) for k, v in ent_dict.items()}

        tgts = []
        for token in doc:
            if token.lemma_.lower() in TARGET_VERBS:
                for child in token.children:
                    if child.dep_ in ("dobj","pobj","attr"):
                        tgts.append({
                            "action":   token.lemma_.lower(),
                            "target":   " ".join(t.text for t in child.subtree),
                            "sentence": token.sent.text.strip(),
                        })
        all_targets[i] = tgts

        lemmas = [t.lemma_ for t in doc
                  if t.is_alpha and not t.is_stop and len(t.lemma_) > 2]
        n = max(len(lemmas), 1)
        all_lex[i] = {
            dim: round(sum(1 for l in lemmas if l in fs) / n, 5)
            for dim, fs in LEXICON_FROZEN.items()
        }

    log.info("Pass 2/3 — DistilBERT on %d entities…", len(flat_ent_texts))
    flat_preds = _batch_sentiment(flat_ent_texts)

    log.info("Pass 3/3 — assembling DataFrame…")
    rows = []
    for i, chunk in enumerate(chunks):
        s, e = entity_ranges[i]
        sentiment = [
            {"entity": ent, "label": pred["label"], "score": round(pred["score"],4)}
            for ent, pred in zip(flat_ent_texts[s:e], flat_preds[s:e])
        ]
        row = {
            "chunk":     chunk,
            "topic":     topics[i],
            "cluster":   int(clusters[i]),
            "section":   i // 10,
            "entities":  all_entities[i],
            "targets":   all_targets[i],
            "sentiment": sentiment,
        }
        row.update(all_lex[i])
        rows.append(row)

    df = pd.DataFrame(rows)
    log.info("Dataset built: %d rows × %d columns.", *df.shape)
    return df

df = build_dataset(chunks, topics, cluster_labels)

---
## 8 · NLI Validation Layer
Validates high-signal chunks using BART-large-MNLI. Adds `nli_<dim>` columns.

In [ ]:
def nli_validate_top_chunks(
    df: pd.DataFrame,
    top_n: int = 50,
    lex_threshold: float = 0.02,
) -> pd.DataFrame:
    for dim in DIMENSIONS:
        df[f"nli_{dim}"] = float("nan")
    df["_lex_sum"] = df[DIMENSIONS].sum(axis=1)
    top_idx    = set(df.nlargest(top_n, "_lex_sum").index)
    thresh_idx = set(df.index[df[DIMENSIONS].gt(lex_threshold).any(axis=1)])
    candidates = sorted(top_idx | thresh_idx)
    log.info("NLI: scoring %d / %d chunks.", len(candidates), len(df))
    pipe = _get_nli_pipeline()
    labels = list(NLI_HYPOTHESES.keys())
    hyps   = [NLI_HYPOTHESES[d] for d in labels]
    hyp_to_dim = {v: k for k, v in NLI_HYPOTHESES.items()}
    for i, idx in enumerate(candidates):
        if i % 10 == 0:
            log.info("  NLI: %d / %d", i, len(candidates))
        result = pipe(df.at[idx, "chunk"][:1024], candidate_labels=hyps, truncation=True)
        for label, score in zip(result["labels"], result["scores"]):
            df.at[idx, f"nli_{hyp_to_dim[label]}"] = round(score, 4)
    df.drop(columns=["_lex_sum"], inplace=True)
    log.info("NLI complete.")
    return df

df = nli_validate_top_chunks(df, top_n=NLI_TOP_N, lex_threshold=NLI_THRESHOLD)

---
## 9 · Network + Targets + Fingerprint

In [ ]:
def build_network(df: pd.DataFrame) -> nx.Graph:
    G = nx.Graph()
    for _, row in df.iterrows():
        ent_pool = []
        for entity_list in row["entities"].values():
            ent_pool.extend(entity_list)
        for e1, e2 in combinations(set(ent_pool), 2):
            if G.has_edge(e1, e2):
                G[e1][e2]["weight"] += 1
            else:
                G.add_edge(e1, e2, weight=1)
    log.info("Network: %d nodes, %d edges.", G.number_of_nodes(), G.number_of_edges())
    return G

def aggregate_targets(df: pd.DataFrame) -> pd.DataFrame:
    all_targets = []
    for target_list in df["targets"]:
        all_targets.extend(target_list)
    if not all_targets:
        return pd.DataFrame(columns=["action","target","sentence","frequency"])
    target_df = pd.DataFrame(all_targets)
    freq = (target_df.groupby(["action","target"]).size()
            .reset_index(name="frequency")
            .sort_values("frequency", ascending=False))
    return target_df.merge(freq, on=["action","target"], how="left")

def build_fingerprint(df: pd.DataFrame) -> dict:
    fp: dict = {"n_chunks": int(len(df))}
    for dim in DIMENSIONS:
        if dim not in df.columns:
            continue
        fp[f"{dim}_lex_mean"] = float(df[dim].mean())
        fp[f"{dim}_lex_max"]  = float(df[dim].max())
        fp[f"{dim}_top_chunk"] = df.loc[df[dim].idxmax(), "chunk"]
        nli_col = f"nli_{dim}"
        if nli_col in df.columns:
            valid = df[nli_col].dropna()
            if not valid.empty:
                fp[f"{dim}_nli_mean"] = float(valid.mean())
                fp[f"{dim}_nli_max"]  = float(valid.max())
                fp[f"{dim}_nli_n"]    = int(len(valid))
    return fp

G          = build_network(df)
target_df  = aggregate_targets(df)
fingerprint = build_fingerprint(df)
log.info("Network, targets, fingerprint complete.")

---
## 10 · Persist CAO  ← **The save point**
Everything downstream reads from here. Run once, report forever.

In [ ]:
# ── Serialise complex columns so parquet/CSV can hold them ───────────────────
df_save = df.copy()
for col in ("entities", "targets", "sentiment"):
    df_save[col] = df_save[col].apply(json.dumps)

df_save.to_parquet(CAO_PARQUET, index=False)
log.info("CAO written: %s  (%d rows)", CAO_PARQUET, len(df_save))

# ── Save meta: fingerprint + network edge list + topic info + targets ────────
meta = {
    "generated": str(datetime.now()),
    "pdf": PDF_PATH,
    "n_chunks": len(df),
    "dimensions": DIMENSIONS,
    "fingerprint": fingerprint,
    "network_nodes": G.number_of_nodes(),
    "network_edges": G.number_of_edges(),
}
with open(CAO_META, "w", encoding="utf-8") as fh:
    json.dump(meta, fh, indent=2)

# Persist supporting artefacts alongside CAO
nx.to_pandas_edgelist(G).to_csv(os.path.join(OUTPUT_DIR, "network_edges.csv"), index=False)
nx.write_graphml(G, os.path.join(OUTPUT_DIR, "network.graphml"))
target_df.to_csv(os.path.join(OUTPUT_DIR, "targets.csv"), index=False)
topic_model_obj.get_topic_info().to_csv(os.path.join(OUTPUT_DIR, "topics_summary.csv"), index=False)

print(f"\n✅ PHASE A COMPLETE")
print(f"   CAO → {CAO_PARQUET}")
print(f"   Meta → {CAO_META}")
print(f"   Proceed to Phase B to generate reports (no reprocessing needed).")

---
# ━━━  PHASE B — REPORT GENERATION (instant, no reprocessing)  ━━━
These cells load the CAO from disk and render reports on demand.
**Start here on any subsequent run.**

---
## 11 · Load CAO from Disk

In [ ]:
# ── Load once; all three tier cells below share this DF ──────────────────────
_df_raw = pd.read_parquet(CAO_PARQUET)

# Deserialise complex columns back to Python objects
for col in ("entities", "targets", "sentiment"):
    _df_raw[col] = _df_raw[col].apply(json.loads)

with open(CAO_META, "r", encoding="utf-8") as fh:
    _meta = json.load(fh)

_targets_full  = pd.read_csv(os.path.join(OUTPUT_DIR, "targets.csv"))
_G_full        = nx.read_graphml(os.path.join(OUTPUT_DIR, "network.graphml"))
_fingerprint   = _meta["fingerprint"]

print(f"✅ CAO loaded: {len(_df_raw)} chunks, {len(_df_raw.columns)} columns")
print(f"   Generated: {_meta['generated']}")
print(f"   Dimensions: {_meta['dimensions']}")

---
## 12 · Tier 1 — Filtered CAO Report
High-signal patterns only: chunks that clear the `TIER1_MIN_LEX_SUM` and `TIER1_MIN_NLI` thresholds.

In [ ]:
def render_tier1(df_full, fingerprint, out_path):
    """Tier 1: high-signal chunks only (lexical + NLI double-filter)."""

    # ── Filter ────────────────────────────────────────────────────────────────
    df = df_full.copy()
    df["_lex_sum"] = df[DIMENSIONS].sum(axis=1)
    mask_lex = df["_lex_sum"] >= TIER1_MIN_LEX_SUM

    nli_cols = [f"nli_{d}" for d in DIMENSIONS if f"nli_{d}" in df.columns]
    if nli_cols:
        mask_nli = df[nli_cols].max(axis=1).fillna(0) >= TIER1_MIN_NLI
        t1 = df[mask_lex & mask_nli].copy()
    else:
        t1 = df[mask_lex].copy()

    t1 = t1.sort_values("_lex_sum", ascending=False)
    log.info("Tier 1: %d / %d chunks pass filter.", len(t1), len(df))

    # ── Build DOCX ────────────────────────────────────────────────────────────
    doc = Document()
    doc.add_heading("Tier 1 — Filtered Narrative Audit", 0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph(
        f"Filter: lex_sum ≥ {TIER1_MIN_LEX_SUM}  |  NLI max ≥ {TIER1_MIN_NLI}  "
        f"→  {len(t1)} chunks shown (of {len(df)} total)"
    )

    # Fingerprint snapshot
    doc.add_heading("Corpus Lexical Fingerprint", level=1)
    fp_table = doc.add_table(rows=1, cols=4)
    fp_table.style = "Table Grid"
    for i, h in enumerate(["Dimension", "Lex Mean", "Lex Max", "NLI Mean"]):
        fp_table.rows[0].cells[i].text = h
    for dim in DIMENSIONS:
        row = fp_table.add_row()
        row.cells[0].text = dim
        row.cells[1].text = f"{fingerprint.get(f'{dim}_lex_mean', 0):.4f}"
        row.cells[2].text = f"{fingerprint.get(f'{dim}_lex_max',  0):.4f}"
        nli_m = fingerprint.get(f"{dim}_nli_mean")
        row.cells[3].text = f"{nli_m:.4f}" if nli_m else "—"

    # Top-signal chunks
    doc.add_heading(f"Top-Signal Chunks (n={min(len(t1), 50)})", level=1)
    for _, r in t1.head(50).iterrows():
        scores = "  ".join(f"{d}={r[d]:.3f}" for d in DIMENSIONS)
        doc.add_heading(f"§{r.name}  sec={r['section']}  cluster={r['cluster']}", level=2)
        doc.add_paragraph(scores)
        doc.add_paragraph(r["chunk"])

    doc.save(out_path)
    log.info("Tier 1 report saved: %s", out_path)
    return t1

T1_PATH = os.path.join(OUTPUT_DIR, "t1_report.docx")
t1_df   = render_tier1(_df_raw, _fingerprint, T1_PATH)
print(f"✅ Tier 1 → {T1_PATH}  ({len(t1_df)} chunks)")

---
## 13 · Tier 2 — Expanded CAO Report
Mid-signal + rhetorical targets + entity sentiment + section heatmap.

In [ ]:
def render_tier2(df_full, targets, fingerprint, out_path):
    """Tier 2: broader filter + targets + entity table + section scores."""

    # ── Filter ────────────────────────────────────────────────────────────────
    df = df_full.copy()
    df["_lex_sum"] = df[DIMENSIONS].sum(axis=1)
    t2 = df[df["_lex_sum"] >= TIER2_MIN_LEX_SUM].sort_values("_lex_sum", ascending=False).copy()
    log.info("Tier 2: %d / %d chunks.", len(t2), len(df))

    # Section aggregation
    section_df = (
        df.groupby("section")[DIMENSIONS].mean().reset_index()
    )

    # Aggregate entity sentiment
    sent_rows = []
    for row in df["sentiment"]:
        sent_rows.extend(row)
    entity_agg = pd.DataFrame()
    if sent_rows:
        sent_df = pd.DataFrame(sent_rows)
        entity_agg = (
            sent_df.groupby("entity")
            .agg(mentions=("entity","count"), mean_score=("score","mean"),
                 dom_label=("label", lambda x: x.mode()[0]))
            .reset_index().sort_values("mentions", ascending=False)
        )

    # ── Build DOCX ────────────────────────────────────────────────────────────
    doc = Document()
    doc.add_heading("Tier 2 — Expanded Narrative Audit", 0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph(
        f"Filter: lex_sum ≥ {TIER2_MIN_LEX_SUM}  →  {len(t2)} of {len(df)} chunks"
    )

    # Fingerprint
    doc.add_heading("1. Lexical Fingerprint (6 Dimensions)", level=1)
    fp_table = doc.add_table(rows=1, cols=4)
    fp_table.style = "Table Grid"
    for i, h in enumerate(["Dimension","Lex Mean","Lex Max","NLI Mean (n)"]):
        fp_table.rows[0].cells[i].text = h
    for dim in DIMENSIONS:
        row = fp_table.add_row()
        row.cells[0].text = dim
        row.cells[1].text = f"{fingerprint.get(f'{dim}_lex_mean', 0):.4f}"
        row.cells[2].text = f"{fingerprint.get(f'{dim}_lex_max',  0):.4f}"
        nli_m, nli_n = fingerprint.get(f"{dim}_nli_mean"), fingerprint.get(f"{dim}_nli_n", 0)
        row.cells[3].text = f"{nli_m:.4f} (n={nli_n})" if nli_m else "—"

    # Section scores
    doc.add_heading("2. Section-Level Scores", level=1)
    sec_t = doc.add_table(rows=1, cols=len(section_df.columns))
    sec_t.style = "Table Grid"
    for i, c in enumerate(section_df.columns):
        sec_t.rows[0].cells[i].text = str(c)
    for _, sr in section_df.iterrows():
        row = sec_t.add_row()
        for i, v in enumerate(sr):
            row.cells[i].text = f"{v:.3f}" if isinstance(v, float) else str(v)

    # Top rhetorical targets
    doc.add_heading("3. Top Rhetorical Targets", level=1)
    if not targets.empty:
        top_t = (
            targets.groupby(["action","target"])["frequency"].max()
            .reset_index().sort_values("frequency", ascending=False)
            .head(TIER2_TOP_TARGETS)
        )
        tgt_table = doc.add_table(rows=1, cols=3)
        tgt_table.style = "Table Grid"
        for i, h in enumerate(["Action","Target","Frequency"]):
            tgt_table.rows[0].cells[i].text = h
        for _, r in top_t.iterrows():
            row = tgt_table.add_row()
            row.cells[0].text = str(r["action"])
            row.cells[1].text = str(r["target"])
            row.cells[2].text = str(r["frequency"])

    # Top entities
    if not entity_agg.empty:
        doc.add_heading("4. Top Named Entities (sentiment)", level=1)
        ent_table = doc.add_table(rows=1, cols=4)
        ent_table.style = "Table Grid"
        for i, h in enumerate(["Entity","Mentions","Mean Score","Dominant Label"]):
            ent_table.rows[0].cells[i].text = h
        for _, r in entity_agg.head(25).iterrows():
            row = ent_table.add_row()
            row.cells[0].text = str(r["entity"])
            row.cells[1].text = str(r["mentions"])
            row.cells[2].text = f"{r['mean_score']:.4f}"
            row.cells[3].text = str(r["dom_label"])

    # Expanded chunks
    doc.add_heading("5. Expanded High-Signal Chunks", level=1)
    for _, r in t2.head(100).iterrows():
        scores = "  ".join(f"{d}={r[d]:.3f}" for d in DIMENSIONS)
        doc.add_heading(f"§{r.name}  sec={r['section']}  cluster={r['cluster']}", level=2)
        doc.add_paragraph(scores)
        # Show NLI scores if present
        nli_vals = {d: r.get(f"nli_{d}") for d in DIMENSIONS}
        nli_str  = "  ".join(f"nli_{d}={v:.3f}" for d, v in nli_vals.items()
                             if v is not None and not (isinstance(v, float) and np.isnan(v)))
        if nli_str:
            doc.add_paragraph(nli_str)
        doc.add_paragraph(r["chunk"])

    doc.save(out_path)
    log.info("Tier 2 report saved: %s", out_path)
    return t2

T2_PATH = os.path.join(OUTPUT_DIR, "t2_report.docx")
t2_df   = render_tier2(_df_raw, _targets_full, _fingerprint, T2_PATH)
print(f"✅ Tier 2 → {T2_PATH}  ({len(t2_df)} chunks)")

---
## 14 · Tier 3 — Full CAO Report
Every chunk. Full NLI table. Network stats. Complete target inventory. No filter.

In [ ]:
def render_tier3(df_full, targets, G, fingerprint, out_path):
    """Tier 3: complete CAO — every chunk, full NLI, network summary."""

    doc = Document()
    doc.add_heading("Tier 3 — Full Canonical Analysis Object", 0)
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph(f"Total chunks: {len(df_full)} | Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # ── Full fingerprint (lex + NLI) ─────────────────────────────────────────
    doc.add_heading("1. Full Lexical + NLI Fingerprint", level=1)
    fp_table = doc.add_table(rows=1, cols=6)
    fp_table.style = "Table Grid"
    for i, h in enumerate(["Dimension","Lex Mean","Lex Max","NLI Mean","NLI Max","NLI n"]):
        fp_table.rows[0].cells[i].text = h
    for dim in DIMENSIONS:
        row = fp_table.add_row()
        row.cells[0].text = dim
        row.cells[1].text = f"{fingerprint.get(f'{dim}_lex_mean', 0):.4f}"
        row.cells[2].text = f"{fingerprint.get(f'{dim}_lex_max',  0):.4f}"
        row.cells[3].text = f"{fingerprint.get(f'{dim}_nli_mean', 0):.4f}" if fingerprint.get(f"{dim}_nli_mean") else "—"
        row.cells[4].text = f"{fingerprint.get(f'{dim}_nli_max',  0):.4f}" if fingerprint.get(f"{dim}_nli_max") else "—"
        row.cells[5].text = str(fingerprint.get(f"{dim}_nli_n", "—"))

    # ── Network summary ───────────────────────────────────────────────────────
    doc.add_heading("2. Entity Co-occurrence Network", level=1)
    doc.add_paragraph(f"Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}")
    # Top 20 nodes by degree
    if G.number_of_nodes() > 0:
        top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:20]
        net_table = doc.add_table(rows=1, cols=2)
        net_table.style = "Table Grid"
        net_table.rows[0].cells[0].text = "Entity"
        net_table.rows[0].cells[1].text = "Degree"
        for node, deg in top_nodes:
            row = net_table.add_row()
            row.cells[0].text = str(node)
            row.cells[1].text = str(deg)

    # ── Full target inventory ─────────────────────────────────────────────────
    doc.add_heading("3. Complete Rhetorical Target Inventory", level=1)
    if not targets.empty:
        top_t = (
            targets.groupby(["action","target"])["frequency"].max()
            .reset_index().sort_values("frequency", ascending=False)
            .head(100)
        )
        tgt_table = doc.add_table(rows=1, cols=3)
        tgt_table.style = "Table Grid"
        for i, h in enumerate(["Action","Target","Frequency"]):
            tgt_table.rows[0].cells[i].text = h
        for _, r in top_t.iterrows():
            row = tgt_table.add_row()
            row.cells[0].text = str(r["action"])
            row.cells[1].text = str(r["target"])
            row.cells[2].text = str(r["frequency"])

    # ── Every chunk ───────────────────────────────────────────────────────────
    doc.add_heading("4. All Chunks — Full Annotation", level=1)
    df_sorted = df_full.copy()
    df_sorted["_lex_sum"] = df_sorted[DIMENSIONS].sum(axis=1)
    df_sorted = df_sorted.sort_values("_lex_sum", ascending=False)

    for _, r in df_sorted.iterrows():
        lex_str = "  ".join(f"{d}={r[d]:.3f}" for d in DIMENSIONS)
        nli_vals = {d: r.get(f"nli_{d}") for d in DIMENSIONS}
        nli_str  = "  ".join(f"nli_{d}={v:.3f}" for d, v in nli_vals.items()
                             if v is not None and not (isinstance(v, float) and np.isnan(v)))
        doc.add_heading(f"§{r.name}  sec={r['section']}  cluster={r['cluster']}", level=2)
        doc.add_paragraph(lex_str)
        if nli_str:
            doc.add_paragraph(nli_str)
        doc.add_paragraph(r["chunk"])

    doc.save(out_path)
    log.info("Tier 3 report saved: %s", out_path)

T3_PATH = os.path.join(OUTPUT_DIR, "t3_report.docx")
render_tier3(_df_raw, _targets_full, _G_full, _fingerprint, T3_PATH)
print(f"✅ Tier 3 → {T3_PATH}  ({len(_df_raw)} chunks, full CAO)")

---
## 15 · Summary

In [ ]:
print("═" * 60)
print("PIPELINE SUMMARY")
print("═" * 60)
print(f"CAO (source of truth)  →  {CAO_PARQUET}")
print(f"CAO meta               →  {CAO_META}")
print()
print(f"Tier 1  (filtered, high-signal)    →  {T1_PATH}")
print(f"         {len(t1_df)} chunks  |  lex_sum ≥ {TIER1_MIN_LEX_SUM}, NLI ≥ {TIER1_MIN_NLI}")
print()
print(f"Tier 2  (expanded, mid-signal)     →  {T2_PATH}")
print(f"         {len(t2_df)} chunks  |  lex_sum ≥ {TIER2_MIN_LEX_SUM}")
print()
print(f"Tier 3  (full CAO, no filter)      →  {T3_PATH}")
print(f"         {len(_df_raw)} chunks")
print()
print("To regenerate any report with different thresholds:")
print("  1. Edit TIER1_MIN_LEX_SUM / TIER1_MIN_NLI / TIER2_MIN_LEX_SUM in cell 1")
print("  2. Run cell 11 (load CAO) then the tier cell you want")
print("  No reprocessing. Instant.")
print("═" * 60)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# COMPLETE MASTER REPORT - All Three Tiers in One Document
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def render_complete_master_report(df_full, targets, G, fingerprint, out_path):
    """
    Creates a single DOCX containing:
    - Executive Summary (Tier 1 concepts)
    - Full Fingerprint
    - Tier 1: High-signal chunks (lex + NLI filtered)
    - Tier 2: Expanded analysis (targets, entities, section scores)
    - Tier 3: Complete CAO (all chunks)
    - Network Analysis
    """
    
    doc = Document()
    
    # ──────────────────────────────────────────────────────────────────────────
    # TITLE PAGE
    # ──────────────────────────────────────────────────────────────────────────
    title = doc.add_heading('COMPLETE NARRATIVE INTELLIGENCE REPORT', 0)
    title.alignment = 1  # Center alignment
    
    doc.add_paragraph()
    doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    doc.add_paragraph(f"Source Document: {Path(PDF_PATH).name}")
    doc.add_paragraph(f"Pages Analyzed: {PAGE_START}–{PAGE_END if PAGE_END else 'END'}")
    doc.add_paragraph(f"Total Chunks: {len(df_full)}")
    doc.add_paragraph(f"Network: {G.number_of_nodes()} entities, {G.number_of_edges()} connections")
    
    doc.add_page_break()
    
    # ──────────────────────────────────────────────────────────────────────────
    # EXECUTIVE SUMMARY (Tier 1 concepts)
    # ──────────────────────────────────────────────────────────────────────────
    doc.add_heading('EXECUTIVE SUMMARY', level=1)
    doc.add_paragraph(
        "This report presents a multi-tiered rhetorical analysis of the source document. "
        "The analysis identifies six rhetorical dimensions: Moral, Threat, Power, Urgency, "
        "Us_vs_Them, and Legitimacy. Each chunk of text is scored on these dimensions using "
        "both lexical matching and Natural Language Inference (NLI) validation."
    )
    
    # Filter high-signal chunks for summary
    df_temp = df_full.copy()
    df_temp["_lex_sum"] = df_temp[DIMENSIONS].sum(axis=1)
    nli_cols = [f"nli_{d}" for d in DIMENSIONS if f"nli_{d}" in df_temp.columns]
    
    if nli_cols:
        mask_lex = df_temp["_lex_sum"] >= TIER1_MIN_LEX_SUM
        mask_nli = df_temp[nli_cols].max(axis=1).fillna(0) >= TIER1_MIN_NLI
        high_signal = df_temp[mask_lex & mask_nli]
    else:
        high_signal = df_temp[df_temp["_lex_sum"] >= TIER1_MIN_LEX_SUM]
    
    doc.add_heading('Key Findings', level=2)
    doc.add_paragraph(f"• {len(high_signal)} high-signal chunks identified (lex_sum ≥ {TIER1_MIN_LEX_SUM}, NLI ≥ {TIER1_MIN_NLI})")
    
    # Top dimensions by mean score
    dim_means = {dim: df_full[dim].mean() for dim in DIMENSIONS}
    top_dim = max(dim_means, key=dim_means.get)
    doc.add_paragraph(f"• Most prevalent rhetorical dimension: {top_dim.upper()} (mean = {dim_means[top_dim]:.4f})")
    
    # Most frequent action targets
    if not targets.empty:
        top_action = targets.groupby("action").size().sort_values(ascending=False).head(3)
        doc.add_paragraph(f"• Top rhetorical actions: {', '.join([f'{k} ({v})' for k, v in top_action.items()])}")
    
    doc.add_page_break()
    
    # ──────────────────────────────────────────────────────────────────────────
    # SECTION 1: FULL FINGERPRINT
    # ──────────────────────────────────────────────────────────────────────────
    doc.add_heading('SECTION 1: FULL LEXICAL + NLI FINGERPRINT', level=1)
    
    fp_table = doc.add_table(rows=1, cols=7)
    fp_table.style = "Table Grid"
    headers = ["Dimension", "Lex Mean", "Lex Max", "NLI Mean", "NLI Max", "NLI n", "Top Chunk (excerpt)"]
    for i, h in enumerate(headers):
        fp_table.rows[0].cells[i].text = h
    
    for dim in DIMENSIONS:
        row = fp_table.add_row()
        row.cells[0].text = dim.upper()
        row.cells[1].text = f"{fingerprint.get(f'{dim}_lex_mean', 0):.4f}"
        row.cells[2].text = f"{fingerprint.get(f'{dim}_lex_max', 0):.4f}"
        row.cells[3].text = f"{fingerprint.get(f'{dim}_nli_mean', 0):.4f}" if fingerprint.get(f"{dim}_nli_mean") else "—"
        row.cells[4].text = f"{fingerprint.get(f'{dim}_nli_max', 0):.4f}" if fingerprint.get(f"{dim}_nli_max") else "—"
        row.cells[5].text = str(fingerprint.get(f"{dim}_nli_n", "—"))
        top_chunk = fingerprint.get(f"{dim}_top_chunk", "")
        row.cells[6].text = top_chunk[:150] + "..." if len(top_chunk) > 150 else top_chunk
    
    doc.add_page_break()
    
    # ──────────────────────────────────────────────────────────────────────────
    # SECTION 2: TIER 1 - HIGH-SIGNAL CHUNKS
    # ──────────────────────────────────────────────────────────────────────────
    doc.add_heading('SECTION 2: TIER 1 — HIGH-SIGNAL PATTERNS', level=1)
    doc.add_paragraph(
        f"Tier 1 includes chunks that clear both thresholds: "
        f"lexical sum ≥ {TIER1_MIN_LEX_SUM} AND NLI confidence ≥ {TIER1_MIN_NLI}. "
        f"These represent the strongest rhetorical signals in the document."
    )
    doc.add_paragraph(f"Total Tier 1 chunks: {len(high_signal)}")
    
    if len(high_signal) > 0:
        high_signal_sorted = high_signal.sort_values("_lex_sum", ascending=False)
        
        for idx, (_, r) in enumerate(high_signal_sorted.iterrows()):
            doc.add_heading(f"Chunk {idx+1} — Section {r['section']}, Topic {r['topic']}, Cluster {r['cluster']}", level=2)
            
            # Lexical scores
            lex_scores = "  ".join(f"{d}={r[d]:.3f}" for d in DIMENSIONS)
            doc.add_paragraph(f"📊 Lexical Scores: {lex_scores}")
            
            # NLI scores if available
            nli_vals = {d: r.get(f"nli_{d}") for d in DIMENSIONS}
            nli_str = "  ".join(f"nli_{d}={v:.3f}" for d, v in nli_vals.items() 
                               if v is not None and not (isinstance(v, float) and np.isnan(v)))
            if nli_str:
                doc.add_paragraph(f"🧠 NLI Validation: {nli_str}")
            
            # Chunk text
            doc.add_paragraph(f"📄 Text: \"{r['chunk']}\"")
            doc.add_paragraph()  # spacing
    else:
        doc.add_paragraph("No chunks met the Tier 1 threshold criteria.")
    
    doc.add_page_break()
    
    # ──────────────────────────────────────────────────────────────────────────
    # SECTION 3: TIER 2 - EXPANDED ANALYSIS
    # ──────────────────────────────────────────────────────────────────────────
    doc.add_heading('SECTION 3: TIER 2 — EXPANDED NARRATIVE ANALYSIS', level=1)
    doc.add_paragraph(
        f"Tier 2 provides broader context with lexical sum ≥ {TIER2_MIN_LEX_SUM}. "
        f"This section includes rhetorical targets, entity sentiment, and section-level scores."
    )
    
    # Filter for Tier 2
    df_temp2 = df_full.copy()
    df_temp2["_lex_sum"] = df_temp2[DIMENSIONS].sum(axis=1)
    tier2 = df_temp2[df_temp2["_lex_sum"] >= TIER2_MIN_LEX_SUM].sort_values("_lex_sum", ascending=False)
    doc.add_paragraph(f"Total Tier 2 chunks: {len(tier2)}")
    
    # 3.1 Section-Level Scores
    doc.add_heading('3.1 Section-Level Rhetorical Scores', level=2)
    section_df = df_full.groupby("section")[DIMENSIONS].mean().reset_index()
    
    sec_table = doc.add_table(rows=1, cols=len(section_df.columns))
    sec_table.style = "Table Grid"
    for i, c in enumerate(section_df.columns):
        sec_table.rows[0].cells[i].text = str(c).upper()
    for _, sr in section_df.iterrows():
        row = sec_table.add_row()
        for i, v in enumerate(sr):
            row.cells[i].text = f"{v:.3f}" if isinstance(v, float) else str(v)
    
    # 3.2 Top Rhetorical Targets
    doc.add_heading('3.2 Top Rhetorical Targets', level=2)
    if not targets.empty:
        top_targets = (
            targets.groupby(["action", "target"])["frequency"].max()
            .reset_index().sort_values("frequency", ascending=False)
            .head(TIER2_TOP_TARGETS)
        )
        
        tgt_table = doc.add_table(rows=1, cols=3)
        tgt_table.style = "Table Grid"
        for i, h in enumerate(["Action", "Target", "Frequency"]):
            tgt_table.rows[0].cells[i].text = h
        for _, r in top_targets.iterrows():
            row = tgt_table.add_row()
            row.cells[0].text = str(r["action"])
            row.cells[1].text = str(r["target"])
            row.cells[2].text = str(r["frequency"])
    else:
        doc.add_paragraph("No rhetorical targets extracted.")
    
    # 3.3 Entity Sentiment
    doc.add_heading('3.3 Named Entity Sentiment Analysis', level=2)
    sent_rows = []
    for row in df_full["sentiment"]:
        sent_rows.extend(row)
    
    if sent_rows:
        sent_df = pd.DataFrame(sent_rows)
        entity_agg = (
            sent_df.groupby("entity")
            .agg(mentions=("entity", "count"), mean_score=("score", "mean"),
                 dom_label=("label", lambda x: x.mode()[0] if len(x) > 0 else "NEUTRAL"))
            .reset_index().sort_values("mentions", ascending=False)
        )
        
        ent_table = doc.add_table(rows=1, cols=4)
        ent_table.style = "Table Grid"
        for i, h in enumerate(["Entity", "Mentions", "Mean Sentiment", "Dominant Label"]):
            ent_table.rows[0].cells[i].text = h
        for _, r in entity_agg.head(25).iterrows():
            row = ent_table.add_row()
            row.cells[0].text = str(r["entity"])
            row.cells[1].text = str(r["mentions"])
            row.cells[2].text = f"{r['mean_score']:.4f}"
            row.cells[3].text = str(r["dom_label"])
    
    # 3.4 Expanded Chunks
    doc.add_heading('3.4 Expanded High-Signal Chunks', level=2)
    for idx, (_, r) in enumerate(tier2.head(75).iterrows()):
        doc.add_heading(f"Chunk {idx+1} — Section {r['section']}, Cluster {r['cluster']}", level=3)
        scores = "  ".join(f"{d}={r[d]:.3f}" for d in DIMENSIONS)
        doc.add_paragraph(scores)
        doc.add_paragraph(r["chunk"])
    
    doc.add_page_break()
    
    # ──────────────────────────────────────────────────────────────────────────
    # SECTION 4: NETWORK ANALYSIS
    # ──────────────────────────────────────────────────────────────────────────
    doc.add_heading('SECTION 4: ENTITY CO-OCCURRENCE NETWORK', level=1)
    doc.add_paragraph(
        f"The entity network maps how named entities co-appear across chunks. "
        f"Nodes represent entities, edges represent co-occurrence in the same chunk, "
        f"and edge weights reflect co-occurrence frequency."
    )
    doc.add_paragraph(f"Network Statistics:")
    doc.add_paragraph(f"• Total Nodes: {G.number_of_nodes()}")
    doc.add_paragraph(f"• Total Edges: {G.number_of_edges()}")
    
    if G.number_of_nodes() > 0:
        # Top 30 nodes by degree
        top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:30]
        
        doc.add_heading('Top 30 Entities (by degree centrality)', level=2)
        net_table = doc.add_table(rows=1, cols=2)
        net_table.style = "Table Grid"
        net_table.rows[0].cells[0].text = "Entity"
        net_table.rows[0].cells[1].text = "Degree (Connections)"
        
        for node, deg in top_nodes:
            row = net_table.add_row()
            row.cells[0].text = str(node)
            row.cells[1].text = str(deg)
    
    doc.add_page_break()
    
    # ──────────────────────────────────────────────────────────────────────────
    # SECTION 5: TIER 3 - COMPLETE CAO (ALL CHUNKS)
    # ──────────────────────────────────────────────────────────────────────────
    doc.add_heading('SECTION 5: TIER 3 — COMPLETE CANONICAL ANALYSIS OBJECT', level=1)
    doc.add_paragraph(
        "Tier 3 presents the complete, unfiltered analysis of every chunk in the document. "
        "No thresholds are applied. Each chunk includes lexical dimension scores and, where available, "
        "NLI validation scores."
    )
    doc.add_paragraph(f"Total chunks in complete CAO: {len(df_full)}")
    
    df_sorted = df_full.copy()
    df_sorted["_lex_sum"] = df_sorted[DIMENSIONS].sum(axis=1)
    df_sorted = df_sorted.sort_values("_lex_sum", ascending=False)
    
    for idx, (_, r) in enumerate(df_sorted.iterrows()):
        doc.add_heading(f"Chunk {idx+1} of {len(df_sorted)} — Section {r['section']}, Topic {r['topic']}, Cluster {r['cluster']}", level=2)
        
        # Lexical scores
        lex_str = "  ".join(f"{d}={r[d]:.3f}" for d in DIMENSIONS)
        doc.add_paragraph(f"📊 Lexical Scores: {lex_str}")
        
        # NLI scores if available
        nli_vals = {d: r.get(f"nli_{d}") for d in DIMENSIONS}
        nli_str = "  ".join(f"nli_{d}={v:.3f}" for d, v in nli_vals.items() 
                           if v is not None and not (isinstance(v, float) and np.isnan(v)))
        if nli_str:
            doc.add_paragraph(f"🧠 NLI Validation: {nli_str}")
        
        # Chunk text
        doc.add_paragraph(f"📄 Text: \"{r['chunk']}\"")
        doc.add_paragraph()  # spacing
    
    # Save the document
    doc.save(out_path)
    log.info("Complete master report saved: %s", out_path)
    return len(df_sorted)


# ──────────────────────────────────────────────────────────────────────────────
# GENERATE THE COMPLETE MASTER REPORT
# ──────────────────────────────────────────────────────────────────────────────
MASTER_REPORT_PATH = os.path.join(OUTPUT_DIR, "complete_master_report.docx")

num_chunks = render_complete_master_report(
    _df_raw, 
    _targets_full, 
    _G_full, 
    _fingerprint, 
    MASTER_REPORT_PATH
)

print(f"\n{'='*70}")
print(f"✅ COMPLETE MASTER REPORT GENERATED")
print(f"{'='*70}")
print(f"📄 File: {MASTER_REPORT_PATH}")
print(f"📊 Contains:")
print(f"   • Executive Summary with key findings")
print(f"   • Section 1: Full Lexical + NLI Fingerprint ({len(DIMENSIONS)} dimensions)")
print(f"   • Section 2: Tier 1 — High-signal chunks (lex_sum ≥ {TIER1_MIN_LEX_SUM}, NLI ≥ {TIER1_MIN_NLI})")
print(f"   • Section 3: Tier 2 — Expanded analysis with targets, entities, section scores")
print(f"   • Section 4: Entity co-occurrence network ({_G_full.number_of_nodes()} nodes, {_G_full.number_of_edges()} edges)")
print(f"   • Section 5: Tier 3 — Complete CAO ({num_chunks} chunks)")
print(f"{'='*70}")